In [9]:
# numpy → 웨이퍼 좌표 계산 및 불량 패턴 생성
# pandas → 각 Die의 위치와 PASS/FAIL 상태를 표 형태로 관리
# matplotlib → 웨이퍼 맵 시각화
# patches → Die 사각형, 웨이퍼 원, Notch 같은 도형 생성
# patheffects → 좌표 글씨의 가독성 향상

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.patches as mpatches

#코드에서 실제로 쓰지 않을 예정이기 때문에 주석으로 두겠음.
#from matplotlib.patches import Circle

import matplotlib.patheffects as pe

In [10]:
def generate_sample_wafer_data(
    wafer_diameter_mm=300.0,
    die_size_mm=10.0,
    edge_exclusion_mm=3.0,
    defect_mode="ring"
):
    """
    실습용 웨이퍼 Die 데이터 생성

    wafer_diameter_mm
    → 웨이퍼 직경
    → 300mm 웨이퍼를 기본값으로 사용

    die_size_mm
    → 한 Die의 크기
    → 정사각형 Die라고 단순 가정

    edge_exclusion_mm
    → 웨이퍼 가장자리에서 분석에서 제외할 영역

    defect_mode
    → 불량 패턴 종류
    → ring / center / scratch / random
    """

    # 웨이퍼 반지름 계산
    radius = wafer_diameter_mm / 2.0

    # 가장자리 제외 영역을 반영한 실제 Die 배치 가능 범위
    effective_radius = radius - edge_exclusion_mm

    # Die 중심 좌표 후보 생성
    coords = np.arange(
        -radius,
        radius + die_size_mm,
        die_size_mm
    )

    die_records = []

    # X, Y 좌표를 하나씩 확인하면서
    # 웨이퍼 원 내부에 들어오는 Die만 저장
    for x in coords:
        for y in coords:

            # 웨이퍼 중심에서 현재 Die까지의 거리
            dist = np.sqrt(
                x**2 + y**2
            )

            # Die 전체가 웨이퍼 안쪽에 들어오는 경우만 사용
            if dist + die_size_mm / 2 <= effective_radius:

                die_records.append({
                    'die_x': x,
                    'die_y': y,

                    # 중심으로부터 거리
                    'radius': dist,

                    # 중심 기준 각도
                    'angle_deg': np.degrees(
                        np.arctan2(y, x)
                    )
                })

    # Die 좌표를 DataFrame으로 변환
    df = pd.DataFrame(die_records)

In [11]:
    # --------------------------------
    # 불량 패턴 적용
    # --------------------------------

    if defect_mode == 'ring':

        # Ring 불량
        # 웨이퍼 가장자리 특정 반경 구간을 FAIL 처리
        df['status'] = np.where(
            (
                (df['radius'] >= 120)
                & (df['radius'] <= 145)
            ),
            'FAIL',
            'PASS'
        )

    elif defect_mode == 'center':

        # Center 불량
        # 웨이퍼 중심 30mm 이내를 FAIL 처리
        df['status'] = np.where(
            df['radius'] <= 30,
            'FAIL',
            'PASS'
        )

    elif defect_mode == 'scratch':

        # Scratch 불량
        # x + y가 특정 범위에 있는 Die를
        # 대각선 스크래치라고 단순 가정
        df['status'] = np.where(
            (
                df['die_x']
                + df['die_y']
            ).between(-15, 15),
            'FAIL',
            'PASS'
        )

    else:

        # Random 불량
        # 전체 Die 중 약 5%를 무작위 FAIL 처리

        np.random.seed(42)

        df['status'] = np.where(
            np.random.random(len(df)) < 0.05,
            'FAIL',
            'PASS'
        )

NameError: name 'defect_mode' is not defined

In [ ]:
    # --------------------------------
    # Yield 계산
    # --------------------------------

    # 전체 Die 개수
    total = len(df)

    # PASS Die 개수
    passed = (
        df['status'] == 'PASS'
    ).sum()

    # Yield 계산
    yield_pct = (
        passed / total * 100
    )

    # DataFrame 자체의 추가 정보로 저장
    df.attrs['yield_pct'] = yield_pct
    df.attrs['total_dies'] = total
    df.attrs['defect_mode'] = defect_mode

    return df

In [ ]:
df = generate_sample_wafer_data()

In [ ]:
def plot_wafer_map(
    df,
    lot_id='LOT_001',
    wafer_id='W01',
    die_size_mm=10.0,
    wafer_radius_mm=150.0,
    ax=None,
    show_die_coords=False
):
    """
    DataFrame에 저장된 Die 상태를
    실제 웨이퍼 맵 형태로 시각화
    """

    # 별도의 Axes가 없으면 새 Figure 생성
    if ax is None:
        fig, ax = plt.subplots(
            figsize=(8, 8)
        )

    # PASS / FAIL 상태별 색상
    color_map = {
        'PASS': '#4CAF50',
        'FAIL': '#F44336'
    }

    # Die 크기의 절반
    half = die_size_mm / 2.0

In [ ]:
    # --------------------------------
    # Die 하나씩 웨이퍼 위에 표시
    # --------------------------------

    for _, row in df.iterrows():

        # PASS / FAIL에 따라 색상 지정
        color = color_map.get(
            row['status'],
            '#9E9E9E'
        )

        # Die를 사각형으로 표현
        rect = plt.Rectangle(
            (
                row['die_x'] - half,
                row['die_y'] - half
            ),

            # Die 사이에 약간의 간격을 보이도록 0.9배
            die_size_mm * 0.9,
            die_size_mm * 0.9,

            linewidth=0.3,
            edgecolor='white',
            facecolor=color,
            alpha=0.85
        )

        ax.add_patch(rect)

        # Die 개수가 적은 경우 좌표까지 표시
        if show_die_coords and len(df) < 200:

            ax.text(
                row['die_x'],
                row['die_y'],

                f"({int(row['die_x'])}, "
                f"{int(row['die_y'])})",

                fontsize=4,
                ha='center',
                va='center',

                color='white',

                path_effects=[
                    pe.withStroke(
                        linewidth=0.5,
                        foreground='black'
                    )
                ]
            )

In [ ]:
    # --------------------------------
    # 웨이퍼 외곽선
    # --------------------------------

    wafer_circle = Circle(
        (0, 0),
        wafer_radius_mm,

        fill=False,
        edgecolor='#455A64',
        linewidth=2.5
    )

    ax.add_patch(wafer_circle)

    # --------------------------------
    # Edge Exclusion 영역
    # --------------------------------

    edge_ex_circle = Circle(
        (0, 0),

        wafer_radius_mm - 3,

        fill=False,
        edgecolor='#B0BEC5',
        linewidth=1.0,
        linestyle='--'
    )

    ax.add_patch(edge_ex_circle)

    # --------------------------------
    # Notch 표시
    # --------------------------------

    notch = plt.Polygon(
        [
            [-5, -wafer_radius_mm],
            [0, -wafer_radius_mm - 8],
            [5, -wafer_radius_mm]
        ],

        closed=True,
        facecolor='#455A64',
        edgecolor='#455A64'
    )

    ax.add_patch(notch)

In [ ]:
    # 웨이퍼가 잘리지 않도록 그래프 범위 설정
    ax.set_xlim(
        -wafer_radius_mm * 1.15,
        wafer_radius_mm * 1.15
    )

    ax.set_ylim(
        -wafer_radius_mm * 1.25,
        wafer_radius_mm * 1.15
    )

    # X/Y 비율을 동일하게 유지
    ax.set_aspect('equal')

    # 배경 설정
    ax.set_facecolor('#ECEFF1')

    # 격자 제거
    ax.grid(False)

    # 축 글자 설정
    ax.tick_params(labelsize=8)

    ax.set_xlabel(
        'X position (mm)',
        fontsize=9
    )

    ax.set_ylabel(
        'Y position (mm)',
        fontsize=9
    )

In [ ]:
    # DataFrame에 저장해둔 수율 정보 불러오기
    yield_pct = df.attrs.get(
        'yield_pct',
        0
    )

    total_dies = df.attrs.get(
        'total_dies',
        len(df)
    )

    fail_dies = (
        df['status'] == 'FAIL'
    ).sum()

    defect_mode = df.attrs.get(
        'defect_mode',
        'unknown'
    )

    # 웨이퍼 정보 제목 표시
    ax.set_title(
        f"Lot: {lot_id} | Wafer: {wafer_id}\n"
        f"Yield: {yield_pct:.1f}% | "
        f"Total: {total_dies} | "
        f"FAIL: {fail_dies} | "
        f"Pattern: {defect_mode.upper()}",

        fontsize=10,
        fontweight='bold',
        pad=12
    )

In [ ]:
    # PASS / FAIL 범례 생성

    legend_elements = [

        mpatches.Patch(
            facecolor='#4CAF50',
            edgecolor='white',
            label=f"PASS ({total_dies - fail_dies})"
        ),

        mpatches.Patch(
            facecolor='#F44336',
            edgecolor='white',
            label=f"FAIL ({fail_dies})"
        )
    ]

    ax.legend(
        handles=legend_elements,

        loc='lower center',

        bbox_to_anchor=(
            0.5,
            -0.13
        ),

        ncol=2,
        fontsize=9,
        framealpha=0.9,
        edgecolor='#B0BEC5'
    )

    return ax

In [ ]:
# Ring 형태의 불량 패턴을 가진
# 샘플 웨이퍼 생성

df_ring = generate_sample_wafer_data(
    defect_mode='ring'
)

print(
    "전체 Die:",
    df_ring.attrs['total_dies']
)

print(
    "Yield:",
    f"{df_ring.attrs['yield_pct']:.2f}%"
)

df_ring.head()

In [ ]:
# Ring 형태 웨이퍼 맵 시각화

fig, ax = plt.subplots(
    figsize=(9, 9)
)

plot_wafer_map(
    df_ring,

    lot_id='SEMI_LOT_001',
    wafer_id='W01',

    ax=ax
)

plt.tight_layout()
plt.show()

In [ ]:
# 웨이퍼 중심부에 불량이 집중된 패턴

df_center = generate_sample_wafer_data(
    defect_mode='center'
)

fig, ax = plt.subplots(
    figsize=(9, 9)
)

plot_wafer_map(
    df_center,
    lot_id='SEMI_LOT_001',
    wafer_id='W02',
    ax=ax
)

plt.tight_layout()
plt.show()

In [ ]:
# 선형 Scratch 형태의 불량 패턴

df_scratch = generate_sample_wafer_data(
    defect_mode='scratch'
)

fig, ax = plt.subplots(
    figsize=(9, 9)
)

plot_wafer_map(
    df_scratch,
    lot_id='SEMI_LOT_001',
    wafer_id='W03',
    ax=ax
)

plt.tight_layout()
plt.show()

In [ ]:
# 특별한 공간 패턴 없이
# 무작위로 발생하는 불량

df_random = generate_sample_wafer_data(
    defect_mode='random'
)

fig, ax = plt.subplots(
    figsize=(9, 9)
)

plot_wafer_map(
    df_random,
    lot_id='SEMI_LOT_001',
    wafer_id='W04',
    ax=ax
)

plt.tight_layout()
plt.show()

Ring
→ 웨이퍼 Edge 쪽에 불량 집중

Center
→ 중심부 불량 집중

Scratch
→ 선 형태 불량

Random
→ 공간적 규칙이 뚜렷하지 않음